In [2]:
import os

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [12]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

It keeps calling the model in a `while True` loop.

Each iteration:
1. Send the full message history to the model.
2. Check the response for any `function_call`s.
3. If there are tool calls, run them and append the results to `messages`.
4. If there are no function calls, `break` out of the loop.

So the stop condition is simple: **when the model returns a response with no function calls, the loop ends.**


In [4]:
import sqlite3
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult, SimpleSpanProcessor

DB_PATH = "/Users/arnenyecknyeck/Desktop/llmzoomcamp26/05-monitoring/traces.db"

class SQLiteSpanExporter(SpanExporter):
    def __init__(self, db_path=DB_PATH):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(SQLiteSpanExporter(DB_PATH)))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("llm-zoomcamp")

In [5]:

from starter import index, client
from rag_helper import RAGBase

class RAGTraced(RAGBase):
    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

    def search(self, query):
        with tracer.start_as_current_span("search"):
            return super().search(query)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm"):
            return super().llm(prompt)

rag = RAGTraced(index=index, llm_client=client)
answer = rag.rag("How does the agentic loop keep calling the model until it stops?")
print(answer) #3 entries

It keeps calling the model in a `while True` loop.

Each iteration:
1. Send the full `messages` history to the model.
2. Check the response for any `function_call` items.
3. Run the tool and append the tool output to `messages`.
4. If there were function calls, loop again.
5. If there were no function calls, `break`.

So the stop condition is simple: **the loop ends when the model returns a response with no function calls**.


In [11]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("/Users/arnenyecknyeck/Desktop/llmzoomcamp26/05-monitoring/traces.db")
df = pd.read_sql("SELECT * FROM spans", conn)
print(df)

     name           start_time             end_time  input_tokens  \
0  search  1784510191714595000  1784510191715280000           NaN   
1     llm  1784510191716165000  1784510193765117000           NaN   
2     rag  1784510191714560000  1784510193765930000           NaN   
3  search  1784510347484084000  1784510347486884000           NaN   
4     llm  1784510347488313000  1784510349309534000        7111.0   
5     rag  1784510347484036000  1784510349312471000           NaN   

   output_tokens      cost  
0            NaN       NaN  
1            NaN       NaN  
2            NaN       NaN  
3            NaN       NaN  
4          108.0  0.001131  
5            NaN       NaN  


In [12]:
df['duration'] = df['end_time'] - df['start_time']
df_filtered = df[df['name'] != 'rag']
print(df_filtered.groupby('name')['duration'].sum())

name
llm       3870173000
search       3485000
Name: duration, dtype: int64


In [13]:

for i in range(3):
    answer = rag.rag("How does the agentic loop keep calling the model until it stops?")
    print(f"Run {i+2} done")

Run 2 done
Run 3 done
Run 4 done


In [14]:
conn = sqlite3.connect("/Users/arnenyecknyeck/Desktop/llmzoomcamp26/05-monitoring/traces.db")
df = pd.read_sql("SELECT * FROM spans", conn)
llm_spans = df[df['name'] == 'llm']['input_tokens'].dropna()
print(llm_spans.values)
print(f"Min: {llm_spans.min()}, Max: {llm_spans.max()}")
print(f"Variation: {(llm_spans.max() - llm_spans.min()) / llm_spans.mean() * 100:.1f}%")

[7111. 7111. 7111. 7111.]
Min: 7111.0, Max: 7111.0
Variation: 0.0%


In [10]:
from starter import index, client
from rag_helper import RAGBase

class RAGTraced(RAGBase):
    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

    def search(self, query):
        with tracer.start_as_current_span("search"):
            return super().search(query)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            usage = response.usage
            input_tokens = usage.input_tokens
            output_tokens = usage.output_tokens
            cost = input_tokens * 0.15 / 1_000_000 + output_tokens * 0.60 / 1_000_000
            span.set_attribute("input_tokens", input_tokens)
            span.set_attribute("output_tokens", output_tokens)
            span.set_attribute("cost", cost)
            return response

rag = RAGTraced(index=index, llm_client=client)
answer = rag.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

It keeps calling the model in a `while True` loop.

Each iteration:

1. Send the current `messages` history to the model.
2. Check the response:
   - if it contains a `function_call`, run the tool, append the tool output, and keep looping;
   - if it contains no function calls, break out of the loop.
3. The loop stops when `has_function_calls == False`.

So the stop condition is simply: **no tool calls in the latest model response**.


In [4]:
## 2
class RAGTraced(RAGBase):
    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

    def search(self, query):
        with tracer.start_as_current_span("search"):
            return super().search(query)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            usage = response.usage
            input_tokens = usage.input_tokens
            output_tokens = usage.output_tokens
            cost = input_tokens * 0.15 / 1_000_000 + output_tokens * 0.60 / 1_000_000
            span.set_attribute("input_tokens", input_tokens)
            span.set_attribute("output_tokens", output_tokens)
            span.set_attribute("cost", cost)
            return response
rag = RAGTraced(index=index, llm_client=client)
answer = rag.rag("How does the agentic loop keep calling the model until it stops?")
print(answer) # 7111 so 7000

{
    "name": "search",
    "context": {
        "trace_id": "0x4f123e93592a41818ba1b95d28fadc41",
        "span_id": "0x8f585930679b7bb4",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xce63ff83a8c7f630",
    "start_time": "2026-07-20T01:01:20.800333Z",
    "end_time": "2026-07-20T01:01:20.806887Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "1375c201-73ec-4dfb-a82b-31aa595c1fda",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x4f123e93592a41818ba1b95d28fadc41",
        "span_id": "0x17046c443b03c49f",
        "trace_state": "[]"
    },
    "kind": "SpanKind

In [ ]:
## Q3 Over 2000ms

In [ ]:
import sqlite3
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult, SimpleSpanProcessor


In [15]:
## Q4

import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult

class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True


In [16]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("traces.db")
df = pd.read_sql("SELECT * FROM spans", conn)
print(df)

Empty DataFrame
Columns: [name, start_time, end_time, input_tokens, output_tokens, cost]
Index: []
